# Evaluation: Turkish-Gemma-9B QLoRA Medical QA

Orchestration notebook for running `src/evaluate.py` on Google Colab.

**Setup:**
- **Code** is cloned from GitHub (single source of truth)
- **Data & adapter** are loaded from Google Drive
- **Colab** serves as stateless GPU compute

This notebook contains no evaluation logic — it only orchestrates the pipeline.

## 1. Clone Repository

In [1]:
import os

REPO_URL = "https://github.com/Ahmetemintek/gemma-finetuning.git"
REPO_DIR = "/content/gemma-finetuning"

if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

Cloning into '/content/gemma-finetuning'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 100 (delta 43), reused 68 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 65.65 KiB | 1.56 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/gemma-finetuning


## 2. Install Dependencies

In [2]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00


## 3. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/gemma-finetuning"
print(f"Drive project dir: {DRIVE_PROJECT_DIR}")

Mounted at /content/drive
Drive project dir: /content/drive/MyDrive/gemma-finetuning


## 4. Set HF Token (Colab Secrets)

In [4]:
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN loaded from Colab Secrets.")

HF_TOKEN loaded from Colab Secrets.


## 5. Sync Validation Dataset from Drive

In [5]:
DRIVE_DATA_DIR = os.path.join(DRIVE_PROJECT_DIR, "data", "processed")
LOCAL_DATA_DIR = os.path.join(REPO_DIR, "data", "processed")

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
!cp -v {DRIVE_DATA_DIR}/validation.jsonl {LOCAL_DATA_DIR}/
!ls -lh {LOCAL_DATA_DIR}/

'/content/drive/MyDrive/gemma-finetuning/data/processed/validation.jsonl' -> '/content/gemma-finetuning/data/processed/validation.jsonl'
total 2.1M
-rw------- 1 root root 2.1M Feb 27 23:44 validation.jsonl


## 6. Sync LoRA Adapter from Drive

In [6]:
DRIVE_CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT_DIR, "outputs", "checkpoints")
LOCAL_CHECKPOINT_DIR = os.path.join(REPO_DIR, "outputs", "checkpoints")

os.makedirs(LOCAL_CHECKPOINT_DIR, exist_ok=True)
!cp -v {DRIVE_CHECKPOINT_DIR}/adapter_model.safetensors {LOCAL_CHECKPOINT_DIR}/
!cp -v {DRIVE_CHECKPOINT_DIR}/adapter_config.json {LOCAL_CHECKPOINT_DIR}/
!cp -v {DRIVE_CHECKPOINT_DIR}/tokenizer_config.json {LOCAL_CHECKPOINT_DIR}/
!cp -v {DRIVE_CHECKPOINT_DIR}/tokenizer.json {LOCAL_CHECKPOINT_DIR}/
!ls -lh {LOCAL_CHECKPOINT_DIR}/

'/content/drive/MyDrive/gemma-finetuning/outputs/checkpoints/adapter_model.safetensors' -> '/content/gemma-finetuning/outputs/checkpoints/adapter_model.safetensors'
'/content/drive/MyDrive/gemma-finetuning/outputs/checkpoints/adapter_config.json' -> '/content/gemma-finetuning/outputs/checkpoints/adapter_config.json'
'/content/drive/MyDrive/gemma-finetuning/outputs/checkpoints/tokenizer_config.json' -> '/content/gemma-finetuning/outputs/checkpoints/tokenizer_config.json'
'/content/drive/MyDrive/gemma-finetuning/outputs/checkpoints/tokenizer.json' -> '/content/gemma-finetuning/outputs/checkpoints/tokenizer.json'
total 239M
-rw------- 1 root root 1.1K Feb 27 23:44 adapter_config.json
-rw------- 1 root root 207M Feb 27 23:44 adapter_model.safetensors
-rw------- 1 root root  489 Feb 27 23:44 tokenizer_config.json
-rw------- 1 root root  33M Feb 27 23:44 tokenizer.json


## 7. Run Evaluation

In [7]:
!python src/evaluate.py

Loading validation dataset...
Generating train split: 820 examples [00:00, 42069.76 examples/s]
Validation samples: 820
Loading tokenizer...
config.json: 100% 852/852 [00:00<00:00, 4.50MB/s]
tokenizer_config.json: 46.7kB [00:00, 19.6MB/s]
tokenizer.json: 100% 17.5M/17.5M [00:01<00:00, 9.10MB/s]
special_tokens_map.json: 100% 636/636 [00:00<00:00, 3.91MB/s]
Loading quantized model (4-bit)...
model.safetensors.index.json: 39.1kB [00:00, 3.47MB/s]
Fetching 4 files: 100% 4/4 [00:51<00:00, 12.77s/it]
Download complete: 100% 18.5G/18.5G [00:51<00:00, 384MB/s]                
Loading weights:   0% 0/464 [00:00<?, ?it/s]
Loading weights:   0% 1/464 [00:00<00:00, 9915.61it/s, Materializing param=model.embed_tokens.weight]
Loading weights:   0% 1/464 [00:00<00:00, 5262.61it/s, Materializing param=model.embed_tokens.weight]
Loading weights:   0% 2/464 [00:01<04:44,  1.62it/s, Materializing param=model.embed_tokens.weight]  
Loading weights:   0% 2/464 [00:01<04:44,  1.62it/s, Materializing param=m

## 8. Inference

In [ ]:
!python src/inference.py

## 9. Push to Huggingface Hub

In [ ]:
!python src/push_to_hub.py